In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import japanize_matplotlib 
import seaborn as sns

root_dir = os.path.dirname(os.getcwd())

df = pd.read_csv('../Data_time_series/en_setai_over2_monthly.csv')
data_file = os.path.join(root_dir,'Data_time_series', 'en_share_over2_monthly.csv')
df_share = pd.read_csv(data_file)

# Filter for rows where Medium (中分類) and Minor (小分類) are '-'
# This isolates the top-level "Major" rows

major_rows = df[ (df['Category level 1'] != '-') & (df['Category level 2'] == '-') & (df['Category level 3'] == '-') ]

major_dict = dict(zip(major_rows['Category level 1'], major_rows['Item']))
print(major_dict['1']) # Output: 食料

hierarchy = {}

# Iterate through every row
for index, row in df.iterrows():
    major_id = row['Category level 1']
    med_id = row['Category level 2']
    min_id = row['Category level 3']
    name = row['Item']

    # Skip header/garbage rows if any
    if major_id == '-': continue 

    # 1. Initialize Major Level
    # We use the ID as the key, but store the Name inside
    if major_id not in hierarchy:
        # Look up the major name from our simple dict
        major_name = major_dict.get(major_id, "他")
        hierarchy[major_id] = {'name': major_name, 'med': {}}

    # 2. Add Medium Level (if this row represents a Medium category or deeper)
    if med_id != '-':
        if med_id not in hierarchy[major_id]['med']:
             # Use the name if this is the defining row, otherwise generic placeholder until found
            hierarchy[major_id]['med'][med_id] = {'name': name, 'small': {}}
        
        # Update name if this is exactly the Medium definition row
        if min_id == '-':
            hierarchy[major_id]['med'][med_id]['name'] = name

    # 3. Add Minor Level (if this row represents a Minor category)
    if min_id != '-':
        # Add the minor category to the medium's children
        hierarchy[major_id]['med'][med_id]['small'][min_id] = name
# Usage:
# hierarchy[1]['children'][1]['name']  -> Access the name of Medium Category 1 inside Major 1
# Initialize the flat dictionary
# Key = Item Name (e.g., 'パン'), Value = Formatted String (e.g., '食料ーパン')

name_to_path_map = {}

# Iterate through the hierarchy
for major_id, major_data in hierarchy.items():
    major_name = major_data['name']
    
    # 1. Add the Major category itself (Optional, if needed)
    # name_to_path_map[major_name] = major_name
    
    # Check if 'med' exists
    if 'med' in major_data:
        for med_id, med_data in major_data['med'].items():
            med_name = med_data['name']
            
            # 2. Add Medium Category: Input '穀類' -> Output '食料ー穀類'
            name_to_path_map[med_name] = f"{major_name}ー{med_name}"
            
            # Check if 'small' exists
            if 'small' in med_data:
                for small_id, small_name in med_data['small'].items():
                    # 3. Add Small Category: Input 'パン' -> Output '食料ーパン'
                    # Note: Using strict user format "Major-Small". 
                    # If you wanted full path "Major-Med-Small", use: f"{major_name}ー{med_name}ー{small_name}"
                    name_to_path_map[small_name] = f"{major_name}ー{med_name}ー{small_name}"

# Check the result
print(name_to_path_map['bread']) 
# Output: '食料ーパン'

def get_category_path(item_name):
    return name_to_path_map.get(item_name, item_name) # Returns "他" if not found

print(get_category_path('bread'))

# --- Step 1: Flatten your hierarchy dictionary ---
# This converts: {'1': {'name': 'food', 'med': {'1': {'name': 'grains', 'small': {'1': 'rice'...
# To: {'rice': 'food', 'bread': 'food', ...}

category_map = {}

for l1_id, l1_info in hierarchy.items():
    l1_name = l1_info['name']
    for l2_id, l2_info in l1_info['med'].items():
        # Option A: Map to Level 1 (Food vs Transport)
        # Option B: Map to Level 2 (Grains vs Meat)
        # Let's map to Level 1 for a high-level systemic view
        for l3_id, l3_name in l2_info['small'].items():
            category_map[l3_name] = l1_name

# Convert to a Series for easy grouping
category_series = pd.Series(category_map)

# Dataframe for 3 levels of category hierarchy

c1 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']=='-') & (df_share['Category level 3']=='-')]
c2 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']!='-') & (df_share['Category level 3']=='-')]
c3 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']!='-') & (df_share['Category level 3']!='-')]

import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from scipy.stats import zscore

def prep_data_for_stl(raw_df: pd.DataFrame, meta_col_count: int = 3) -> pd.DataFrame:
    """
    Cleans raw DataFrame and transposes it into a Time x Items format.
    """
    # 1. Extract and Transpose
    df_ts = raw_df.set_index('Item').iloc[:, meta_col_count:]
    df_t = df_ts.T

    # 2. Index Formatting
    df_t.index = pd.to_datetime(df_t.index)
    df_t = df_t.sort_index()

    # 3. Type Casting & Handle Missing Data (STL requires no NaNs)
    df_t = df_t.astype(np.float64)
    # df_t = df_t.ffill().bfill() # Forward/backward fill any missing percentages
    
    return df_t

def get_residuals(series: pd.Series, period: int = 12) -> pd.Series:
    """
    Extracts the residual component from a time series using STL.
    """
    # robust=True handles outliers (like COVID-19 anomalies) better
    res = STL(series, period=period, robust=True).fit()
    return res.resid

# --- Execution Pipeline ---

# STEP 1: Clean and Transpose raw consumption data
df_clean = prep_data_for_stl(c2, meta_col_count=3)

# STEP 2: Apply STL to get the Residuals (Deviations from trend/seasonality)
df_residuals = df_clean.apply(get_residuals, period=12)

# STEP 3: Apply Min-Max Normalization
df_res_norm = df_residuals.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)

# STEP 4: Output as Markdown
# Using .to_markdown() converts the dataframe head into a formatted markdown table
markdown_output = df_res_norm.head().to_markdown()
print(markdown_output)
print(df_res_norm.shape)

food
foodーgrainsーbread
foodーgrainsーbread
|                     |   grains |   seafood |     meat |   milk eggs |   Vegetables/seaweed |    fruit |   Oils and seasonings |   Confectionery |   cooked food |   beverage |   Alcoholic beverages |   Eating out |   rent and ground rent |   Equipment repair/maintenance |   electricity bill |   gas bill |   other light heat |   Water and sewage charges |   household durable goods |   Interior equipment/decoration items |   Bedding |   Housework miscellaneous goods |   Housework consumables |   housekeeping services |   Japanese clothes |   clothes |   Shirts/Sweaters |   Underwear |   Fabric/thread |   other clothing |   footwear |   Clothing related services |   pharmaceuticals |   Intakes for maintaining health |   Health and medical supplies/equipment |   health and medical services |   traffic |   Car-related expenses |   communication |   Tuition fees etc. |   Textbooks/study reference materials |   supplementary education |   Educational 

In [12]:
import numpy as np
import pandas as pd
import pywt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from prophet import Prophet
from scipy.signal import hilbert
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

# PyEMD (または emd-signal) のインポートは環境に合わせて調整してください
try:
    from PyEMD import EMD
except ImportError:
    import emd

def analyze_edmd(series, period=12):
    """ EDMDによるKoopman固有値解析 """
    y = series.values
    X = y[:-1].reshape(-1, 1)
    Y = y[1:].reshape(-1, 1)
    
    poly = PolynomialFeatures(degree=2)
    Phi_X = poly.fit_transform(X)
    Phi_Y = poly.transform(Y)
    
    K, _, _, _ = np.linalg.lstsq(Phi_X, Phi_Y, rcond=None)
    eigenvalues = np.linalg.eigvals(K.T)
    
    y_pred = Phi_X @ K
    recon_error = np.std(Y[:, 0] - y_pred[:, 1]) 
    
    return recon_error, np.max(np.abs(eigenvalues))

def evaluate_comprehensive_comparison(series, period=12):
    """
    提案手法と比較手法を統合評価する関数
    """
    results = {}
    y_val = series.values
    t_idx = np.arange(len(y_val))

    # --- 1. 提案手法: STL + Hilbert ---
    res_stl = STL(series, period=period, robust=True).fit()
    s_plus_r = series - res_stl.trend 
    z_signal = hilbert(s_plus_r - s_plus_r.mean())
    unwrapped_phase = np.unwrap(np.angle(z_signal)) / (2 * np.pi)
    phase_r2 = r2_score(unwrapped_phase, np.poly1d(np.polyfit(t_idx, unwrapped_phase, 1))(t_idx))
    
    results['STL + Hilbert (Proposed)'] = {
        'Category': 'C. Dynamical Systems',
        'Residual_STD': res_stl.resid.std(),
        'Phase_Linearity_R2': phase_r2,
        'Dynamical_Stability': 'High (Limit Cycle)'
    }

    # --- 2. 信号処理: Wavelet ---
    coeffs = pywt.wavedec(y_val, 'db4', level=4)
    refined_coeffs = [np.zeros_like(c) for c in coeffs]
    refined_coeffs[1] = coeffs[1] # cD4 (Approx 16 months)
    refined_coeffs[2] = coeffs[2] # cD3 (Approx 8 months)
    y_wavelet_seasonal = pywt.waverec(refined_coeffs, 'db4')[:len(y_val)]
    
    results['Wavelet'] = {
        'Category': 'B. Signal Processing',
        'Residual_STD': np.std(y_val - y_wavelet_seasonal),
        'Phase_Linearity_R2': np.nan,
        'Dynamical_Stability': 'N/A'
    }

    # --- 3. 信号処理: EMD ---
    try:
        # PyEMDの場合
        imfs = EMD().emd(y_val)
        imf_corr = [np.corrcoef(imf, np.sin(2*np.pi*t_idx/period))[0,1] for imf in imfs]
        best_imf = imfs[np.argmax(np.abs(imf_corr))]
        results['EMD'] = {
            'Category': 'B. Signal Processing',
            'Residual_STD': np.std(y_val - best_imf),
            'Phase_Linearity_R2': r2_score(np.unwrap(np.angle(hilbert(best_imf))), np.poly1d(np.polyfit(t_idx, np.unwrap(np.angle(hilbert(best_imf))), 1))(t_idx)),
            'Dynamical_Stability': 'N/A'
        }
    except:
        pass

    # --- 4. 力学系: EDMD ---
    edmd_recon_err, edmd_eig = analyze_edmd(series)
    results['EDMD (Koopman)'] = {
        'Category': 'C. Dynamical Systems',
        'Residual_STD': edmd_recon_err,
        'Phase_Linearity_R2': np.nan,
        'Dynamical_Stability': f'Max Eigenvalue: {edmd_eig:.3f}'
    }

    # --- 5. 統計的予測: Prophet ---
    df_p = pd.DataFrame({'ds': series.index, 'y': y_val})
    m = Prophet(yearly_seasonality=True).fit(df_p)
    forecast = m.predict(df_p)
    results['Prophet'] = {
        'Category': 'A. Statistical Forecasting',
        'Residual_STD': np.std(y_val - forecast['yhat'].values),
        'Phase_Linearity_R2': np.nan,
        'Dynamical_Stability': 'N/A'
    }

    # --- 6. 統計的予測: Kalman Filter (Structural Time Series) ---
    try:
        ssm_model = sm.tsa.UnobservedComponents(series, level='local linear trend', seasonal=12)
        res_ssm = ssm_model.fit(disp=False)
        seasonal_ssm = res_ssm.seasonal.smoothed
        z_ssm = hilbert(seasonal_ssm - np.mean(seasonal_ssm))
        phase_ssm = np.unwrap(np.angle(z_ssm)) / (2 * np.pi)
        phase_r2_ssm = r2_score(phase_ssm, np.poly1d(np.polyfit(t_idx, phase_ssm, 1))(t_idx))

        results['Kalman Filter (SSM)'] = {
            'Category': 'A. Statistical Forecasting',
            'Residual_STD': res_ssm.resid.std(),
            'Phase_Linearity_R2': phase_r2_ssm,
            'Dynamical_Stability': 'Probabilistic'
        }
    except:
        pass

    df_results = pd.DataFrame(results).T
    # Categoryでソート
    df_results = df_results.sort_values('Category')
    return df_results

# 実行例 (df_clean['beverage'] にデータが入っている想定)
comparison_df = evaluate_comprehensive_comparison(df_clean['beverage'])
display(comparison_df)

11:43:06 - cmdstanpy - INFO - Chain [1] start processing
11:43:08 - cmdstanpy - INFO - Chain [1] done processing
/Users/hajimekoike/miniforge3/envs/consumption_data/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


,Category,Residual_STD,Phase_Linearity_R2,Dynamical_Stability
Prophet,A. Statistical Forecasting,0.053608,NaN,N/A
Kalman Filter (SSM),A. Statistical Forecasting,0.12636,0.999969,Probabilistic
Wavelet,B. Signal Processing,0.228732,NaN,N/A
EMD,B. Signal Processing,0.22697,0.999969,N/A
STL + Hilbert (Proposed),C. Dynamical Systems,0.044048,0.999955,High (Limit Cycle)
EDMD (Koopman),C. Dynamical Systems,0.146588,NaN,Max Eigenvalue: 1.000


In [4]:
import os
import numpy as np
import pandas as pd
import pywt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from prophet import Prophet
from scipy.signal import hilbert
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

# EMD library
try:
    from PyEMD import EMD
except ImportError:
    pass

# --- 1. Stuart-Landau Oscillator Fitting ---
def fit_slo(z_signal, dt=1.0):
    """
    Hilbert変換された解析信号からSLOのパラメータ(μ, β, ω)を推定する
    """
    amp = np.abs(z_signal)
    phase = np.unwrap(np.angle(z_signal))
    
    # 2. omega = mean(d_phase/dt)
    d_phase = np.gradient(phase, dt)
    omega = np.mean(d_phase)
    
    # 3. A_st^2 = mean(A^2)
    A_st_sq = np.mean(amp**2)
    
    if A_st_sq == 0:
        return np.nan, np.nan, omega
        
    # 4. Estimate mu by regression: (dA/dt)/A = mu * (1 - A^2 / A_st^2)
    d_amp = np.gradient(amp, dt)
    Y = d_amp / amp
    X = 1.0 - (amp**2 / A_st_sq)
    
    # Linear regression through origin: Y = mu * X -> mu = (X^T X)^-1 X^T Y
    mu = np.dot(X, Y) / np.dot(X, X) if np.dot(X,X) != 0 else np.nan
    beta = mu / A_st_sq if not np.isnan(mu) else np.nan
    
    return mu, beta, omega

# --- 2. Evaluation Core Function ---
def evaluate_series(series, period=12):
    """ 単一時系列に対する全手法の適用と結果返却 """
    results = {}
    y_val = series.values
    t_idx = np.arange(len(y_val))

    # 1. Proposed: STL + Hilbert + SLO
    try:
        res_stl = STL(series, period=period, robust=True).fit()
        s_plus_r = series - res_stl.trend 
        z_signal = hilbert(s_plus_r - s_plus_r.mean())
        
        phase_r2 = r2_score(np.unwrap(np.angle(z_signal)) / (2 * np.pi), 
                            np.poly1d(np.polyfit(t_idx, np.unwrap(np.angle(z_signal)) / (2 * np.pi), 1))(t_idx))
        
        mu, beta, omega = fit_slo(z_signal)
        
        results['Proposed_Res_STD'] = res_stl.resid.std()
        results['Proposed_Phase_R2'] = phase_r2
        results['SLO_mu (Resilience)'] = mu
        results['SLO_beta'] = beta
        results['SLO_omega'] = omega
    except:
        results.update({'Proposed_Res_STD': np.nan, 'Proposed_Phase_R2': np.nan, 'SLO_mu (Resilience)': np.nan, 'SLO_beta': np.nan, 'SLO_omega': np.nan})

    # 2. State-Space (Kalman Filter)
    try:
        ssm = sm.tsa.UnobservedComponents(series, level='local linear trend', seasonal=12).fit(disp=False)
        results['SSM_Res_STD'] = ssm.resid.std()
    except:
        results['SSM_Res_STD'] = np.nan

    # 3. Wavelet
    try:
        coeffs = pywt.wavedec(y_val, 'db4', level=4)
        refined = [np.zeros_like(c) for c in coeffs]
        refined[1], refined[2] = coeffs[1], coeffs[2] # cD4, cD3
        y_wave = pywt.waverec(refined, 'db4')[:len(y_val)]
        results['Wavelet_Res_STD'] = np.std(y_val - y_wave)
    except:
        results['Wavelet_Res_STD'] = np.nan

    # 4. EDMD
    try:
        X, Y = y_val[:-1].reshape(-1,1), y_val[1:].reshape(-1,1)
        poly = PolynomialFeatures(degree=2)
        Phi_X, Phi_Y = poly.fit_transform(X), poly.transform(Y)
        K = np.linalg.lstsq(Phi_X, Phi_Y, rcond=None)[0]
        results['EDMD_Max_Eigenval'] = np.max(np.abs(np.linalg.eigvals(K.T)))
    except:
        results['EDMD_Max_Eigenval'] = np.nan
        
    return results

# --- 3. Execution Pipeline for All Levels ---
def run_pipeline(c1, c2, c3, meta_col_count=3):
    all_results = []
    
    # Process each hierarchical level
    for df_level, level_name in zip([c1, c2, c3], ['Major(c1)', 'Medium(c2)', 'Minor(c3)']):
        # Clean and transpose
        df_t = df_level.set_index('Item').iloc[:, meta_col_count:].T
        df_t.index = pd.to_datetime(df_t.index)
        df_t = df_t.astype(np.float64).dropna(axis=1) # Drop items with NaNs
        
        for item_name, series in df_t.items():
            metrics = evaluate_series(series)
            metrics['Item'] = item_name
            metrics['Level'] = level_name
            all_results.append(metrics)
            
    # Compile final DataFrame
    final_df = pd.DataFrame(all_results)
    # Reorder columns for readability
    cols = ['Item', 'Level', 'SLO_mu (Resilience)', 'SLO_beta', 'SLO_omega', 
            'Proposed_Phase_R2', 'Proposed_Res_STD', 'SSM_Res_STD', 'Wavelet_Res_STD', 'EDMD_Max_Eigenval']
    return final_df[cols]

# 実行
final_comparison_df = run_pipeline(c1, c2, c3)
display(final_comparison_df)

/Users/hajimekoike/miniforge3/envs/consumption_data/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/hajimekoike/miniforge3/envs/consumption_data/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/hajimekoike/miniforge3/envs/consumption_data/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/hajimekoike/miniforge3/envs/consumption_data/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/hajimekoike/minif

,Item,Level,SLO_mu (Resilience),SLO_beta,SLO_omega,Proposed_Phase_R2,Proposed_Res_STD,SSM_Res_STD,Wavelet_Res_STD,EDMD_Max_Eigenval
0,food,Major(c1),-0.035364,-0.007240,0.986739,0.997521,0.438246,2.381270,2.022516,1.0
1,residence,Major(c1),-0.008133,-0.008941,0.885469,0.997867,0.406072,0.783378,0.586631,1.0
2,Utilities/Water,Major(c1),0.003377,0.001028,0.544796,0.999489,0.293043,0.802677,0.873404,1.0
3,Furniture/household supplies,Major(c1),0.002295,0.004352,0.902020,0.996466,0.228676,0.428630,0.472201,1.0
4,clothing and footwear,Major(c1),0.029651,0.051053,1.068829,0.999550,0.179506,0.577284,0.795453,1.0
...,...,...,...,...,...,...,...,...,...,...
111,personal items,Minor(c3),-0.002216,-0.142874,1.010041,0.994238,0.068207,0.118622,0.086947,1.0
112,Tobacco,Minor(c3),0.000043,0.008434,0.648593,0.990744,0.041400,0.060732,0.051268,1.0
113,Other miscellaneous expenses,Minor(c3),0.001473,0.006322,1.025039,0.997901,0.270458,0.463777,0.621662,1.0
114,gift money,Minor(c3),0.017335,0.006510,0.788553,0.996054,0.252276,0.659200,1.366386,1.0


In [7]:
final_comparison_df.describe()

,SLO_mu (Resilience),SLO_beta,SLO_omega,Proposed_Phase_R2,Proposed_Res_STD,SSM_Res_STD,Wavelet_Res_STD,EDMD_Max_Eigenval
count,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,1.160000e+02
mean,0.002878,0.821035,0.873787,0.996060,0.106647,0.263973,0.325605,1.000000e+00
std,0.047097,8.070078,0.211372,0.004190,0.122653,0.365627,0.406034,1.151496e-14
min,-0.094514,-12.457639,0.524797,0.975905,0.003624,0.007618,0.010117,1.000000e+00
25%,-0.006750,-0.412810,0.671026,0.995412,0.023755,0.055746,0.069094,1.000000e+00
50%,0.000266,0.003595,0.905252,0.997128,0.052775,0.126812,0.174162,1.000000e+00
75%,0.010063,0.153492,1.026478,0.998484,0.155271,0.297321,0.424252,1.000000e+00
max,0.427124,70.657695,1.486402,0.999973,0.543249,2.381270,2.294006,1.000000e+00
